# LangChain: Evaluation
已过时，新版本评估推荐用LangSmith SDK ,用法见官方说明（https://docs.langchain.com/langsmith/evaluation）
下面代码用老方法兼容包改写，能正常运行，之前课程里也用过评估工具，都是类似的，做了解即可，项目用到再具体根据官网来学习即可，学习阶段会一种即可
## Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation
* LangChain evaluation platform

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

## Create our QandA application

In [2]:

from langchain_openai import ChatOpenAI  # 导入模型和嵌入
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader    # 导入文档加载器
from langchain_community.vectorstores import DocArrayInMemorySearch # 导入内存向量存储
from langchain_classic.chains import RetrievalQA

In [3]:
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(
    file_path=file,
    # 强制指定编码为 UTF-8，解决 UnicodeDecodeError
    encoding='utf-8' 
)
data = loader.load()

In [4]:

# model_name：指定要加载的模型名称 (例如 BGE-Small)
# model_kwargs：配置模型加载参数，如 device="cpu" 或 "cuda"
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"}  # 注意：参数名统一为小写
)

# 创建向量数据库 (取代 VectorstoreIndexCreator.from_loaders)
# 直接从 docs 和 embeddings 创建内存向量数据库
db = DocArrayInMemorySearch.from_documents(
    data, 
    embeddings
)

# 获取检索器
retriever = db.as_retriever()

c:\Users\bangsun\miniconda3\envs\jupterlab\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\bangsun\miniconda3\envs\jupterlab\Lib\site-packages\pydantic\_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [ ]:
llm = ChatOpenAI(
    temperature=0.0, 
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY")
    )
# 此方法已经过时，对应新方法见L4-RAG
# 创建RAG问答系统
qa = RetrievalQA.from_chain_type(
    llm=llm,  # 使用的大语言模型
    chain_type="stuff",  # 将所有检索结果"塞给"LLM的模式
    retriever=retriever,  # 使用的检索器
    verbose=True,  # 根据规范，调试阶段开启详细日志
    chain_type_kwargs={
        "document_separator": "<<<<>>>>>"  # 检索结果之间的分隔符
    }
)

### Coming up with test datapoints

In [6]:
data[10]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [7]:
data[11]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

### Hard-coded examples

In [30]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

### LLM-Generated examples

In [ ]:
# 使用LLM自动生成测试用例（扩展测试覆盖）
from langchain_classic.evaluation.qa import QAGenerateChain


In [ ]:
# 创建问题生成链，用于基于文档生成问答对
example_gen_chain = QAGenerateChain.from_llm(llm)

In [ ]:
# the warning below can be safely ignored

In [ ]:
# 为前5个文档生成测试用例
# 注意：根据历史经验，可能会有警告，但通常可以安全忽略
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

verbose=False prompt=PromptTemplate(input_variables=['doc'], input_types={}, partial_variables={}, template='You are a teacher coming up with questions to ask on a quiz.\nGiven the following document, please generate a question and answer based on that document.\n\nExample Format:\n<Begin Document>\n...\n<End Document>\nQUESTION: question here\nANSWER: answer here\n\nThese questions should be detailed and be based explicitly on information in the document. Begin!\n\n<Begin Document>\n{doc}\n<End Document>') llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002A2A2347B50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002A2A2387950>, root_client=<openai.OpenAI object at 0x000002A29F426890>, root_async_client=<openai.AsyncOpenAI object at 0x000002A2A2386F10>, model_name='qwen-max', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://dashscope.aliyuncs

C:\Users\bangsun\AppData\Local\Temp\ipykernel_43484\2433367589.py:2: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(
c:\Users\bangsun\miniconda3\envs\jupterlab\Lib\site-packages\langchain_openai\chat_models\base.py:401: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [ ]:
new_examples[0]


{'qa_pairs': {'query': "What specific features of the Women's Campside Oxfords contribute to their comfort and broken-in feel from the first wear?",
  'answer': "The Women's Campside Oxfords provide an ultracomfortable, broken-in feel from the first wear due to several features: super-soft canvas material, thick cushioning, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, and a moderate arch contour. Additionally, the EVA foam midsole offers cushioning and support, enhancing overall comfort."}}

In [14]:
data[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

### Combine examples

In [ ]:
# 合并手动和自动生成的测试用例
new_examples_clean = []
for item in new_examples:
    # 确保每个 item 都有 'qa_pairs' 键，并且其值包含 'query' 和 'answer'
    qa_pair = item.get('qa_pairs')
    if qa_pair and 'query' in qa_pair and 'answer' in qa_pair:
        new_examples_clean.append({
            "query": qa_pair["query"],
            "answer": qa_pair["answer"]
        })
        
examples += new_examples_clean

print(examples)

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?', 'answer': 'Yes'}, {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?', 'answer': 'The DownTek collection'}, {'query': "What features does the innersole of the Women's Campside Oxfords include to enhance comfort and hygiene?", 'answer': "The innersole of the Women's Campside Oxfords includes comfortable EVA material, Cleansport NXT® antimicrobial odor control, a vintage hunt, fish, and camping motif, and a moderate arch contour."}, {'query': 'What are the dimensions of the Small and Medium Recycled Waterhog Dog Mat, Chevron Weave, and what is one of the key environmental benefits of using this mat?', 'answer': 'The dimensions for the Small Recycled Waterhog Dog Mat, Chevron Weave, are 18" x 28", and for the Medium, they are 22.5" x 34.5". One of the key environmental benefits of using this mat is that it helps keep plastic out of landfills, trails, and oceans by being constru

In [24]:
qa.run(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


'Yes, the Cozy Comfort Pullover Set does have side pockets on the pull-on pants.'

## Manual Evaluation

In [ ]:
# 根据"LangChain日志输出规范"，调试时可开启详细日志
# 手动评估：查看系统如何回答第一个问题
import langchain
langchain.debug = True

In [18]:
qa.run(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


'Yes, the Cozy Comfort Pullover Set does have side pockets. Specifically, the pull-on pants that come with the set feature side pockets along with a wide elastic waistband and drawstring, and a modern slim leg.'

In [19]:
# Turn off the debug mode
langchain.debug = False

## LLM assisted evaluation

In [ ]:
# LLM辅助评估：让另一个LLM作为"评分老师"
# 1. 让系统回答所有测试问题
predictions = qa.apply(examples)

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?', 'answer': 'Yes'}, {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?', 'answer': 'The DownTek collection'}, {'query': "What features does the innersole of the Women's Campside Oxfords include to enhance comfort and hygiene?", 'answer': "The innersole of the Women's Campside Oxfords includes comfortable EVA material, Cleansport NXT® antimicrobial odor control, a vintage hunt, fish, and camping motif, and a moderate arch contour."}, {'query': 'What are the dimensions of the Small and Medium Recycled Waterhog Dog Mat, Chevron Weave, and what is one of the key environmental benefits of using this mat?', 'answer': 'The dimensions for the Small Recycled Waterhog Dog Mat, Chevron Weave, are 18" x 28", and for the Medium, they are 22.5" x 34.5". One of the key environmental benefits of using this mat is that it helps keep plastic out of landfills, trails, and oceans by being constru

In [33]:
# 2. 创建评估链（评分老师）
from langchain_classic.evaluation.qa import QAEvalChain

In [34]:
# 2. 创建评估链（评分老师）
eval_chain = QAEvalChain.from_llm(llm)

In [35]:
# 3. 进行评估，比较系统回答与标准答案
graded_outputs = eval_chain.evaluate(examples, predictions)

c:\Users\bangsun\miniconda3\envs\jupterlab\Lib\site-packages\langchain_openai\chat_models\base.py:401: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [38]:
# 4. 打印评估结果
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['results'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set does have side pockets. The description mentions that the pull-on pants feature side pockets along with a wide elastic waistband and drawstring, and a modern slim leg.
Predicted Grade: GRADE: CORRECT

The student's answer is factually accurate. The true answer is "Yes," and the student correctly stated that the Cozy Comfort Pullover Set does have side pockets, providing additional details that do not conflict with the true answer.

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Predicted Grade: GRADE: CORRECT

The student's answer correctly identifies that the Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection. The additional inform

In [37]:
graded_outputs[0]

{'results': 'GRADE: CORRECT\n\nThe student\'s answer is factually accurate. The true answer is "Yes," and the student correctly stated that the Cozy Comfort Pullover Set does have side pockets, providing additional details that do not conflict with the true answer.'}